# Домашнее задание 3. Парсинг, Git и тестирование на Python

**Цели задания:**

* Освоить базовые подходы к web-scraping с библиотеками `requests` и `BeautisulSoup`: навигация по страницам, извлечение HTML-элементов, парсинг.
* Научиться автоматизировать задачи с использованием библиотеки `schedule`.
* Попрактиковаться в использовании Git и оформлении проектов на GitHub.
* Написать и запустить простые юнит-тесты с использованием `pytest`.


В этом домашнем задании вы разработаете систему для автоматического сбора данных о книгах с сайта [Books to Scrape](http://books.toscrape.com). Нужно реализовать функции для парсинга всех страниц сайта, извлечения информации о книгах, автоматического ежедневного запуска задачи и сохранения результата.

Важной частью задания станет оформление проекта: вы создадите репозиторий на GitHub, оформите `README.md`, добавите артефакты (код, данные, отчеты) и напишете базовые тесты на `pytest`.



### Статус этой копии

Технические ячейки выполнены. Данные и отчёты тестов находятся в `artifacts/`; сводка — в `VALIDATION.md`.

Техническая часть подготовлена с помощью Codex; см. `AI_USAGE.md`. Личная рефлексия заполняется студентом.


In [1]:
# Зависимости устанавливаются командой bash scripts/prepare.sh из корня проекта.
# Эта диагностика ничего не устанавливает и не скрывает отсутствующие модули.
from importlib.util import find_spec
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "scraper.py").is_file(), "Откройте ноутбук из папки проекта"
sys.path.insert(0, str(ROOT))
print("Python:", sys.version.split()[0])
for name in ["requests", "bs4", "schedule", "pytest"]:
    print(f"{name}: {'установлен' if find_spec(name) else 'не установлен'}")

Python: 3.14.2
requests: установлен
bs4: установлен
schedule: установлен
pytest: установлен


In [2]:
import logging
import json
import re
import time
from datetime import datetime, timedelta
from urllib.parse import urljoin

import requests
import schedule
from bs4 import BeautifulSoup

from scraper import (
    BASE_URL, RATINGS, LOGGER, create_session, get_soup, save_books,
)

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s %(message)s", force=True
)


## Задание 1. Сбор данных об одной книге (2 баллов)

В этом задании мы начнем подготовку скрипта для парсинга информации о книгах со страниц каталога сайта [Books to Scrape](https://books.toscrape.com/).

Для начала реализуйте функцию `get_book_data`, которая будет получать данные о книге с одной страницы (например, с [этой](http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html)). Соберите всю информацию, включая название, цену, рейтинг, количество в наличии, описание и дополнительные характеристики из таблицы Product Information. Результат достаточно вернуть в виде словаря.

**Не забывайте про соблюдение PEP-8** — помимо качественно написанного кода важно также документировать функции по стандарту:
* кратко описать, что она делает и для чего нужна;
* какие входные аргументы принимает, какого они типа и что означают по смыслу;
* аналогично описать возвращаемые значения.

*P. S. Состав, количество аргументов функции и тип возвращаемого значения можете менять как вам удобно. То, что написано ниже в шаблоне — лишь пример.*

In [3]:
def get_book_data(
    book_url: str, session: requests.Session | None = None
) -> dict:
    """Собрать полную карточку книги.

    Args:
        book_url: Абсолютный URL страницы книги.
        session: Сессия для повторного использования соединений или None.

    Returns:
        Словарь с title (str), price (float, GBP), rating (int, 1–5),
        availability (int), description (str), product_information (dict),
        currency (str) и url (str). Нет описания — возвращается пустая строка.

    Raises:
        requests.RequestException: Не удалось загрузить страницу.
        ValueError: В HTML отсутствуют обязательные данные или они некорректны.
    """
    if session is None:
        with create_session() as owned_session:
            return get_book_data(book_url, session=owned_session)

    soup = get_soup(book_url, session)
    main = soup.select_one(".product_main")
    table = soup.select_one("table.table-striped")
    if main is None or table is None:
        raise ValueError(f"Нет карточки книги или таблицы: {book_url}")

    title = main.select_one("h1")
    price = main.select_one(".price_color")
    stars = main.select_one(".star-rating")
    stock = main.select_one(".availability")
    if any(element is None for element in (title, price, stars, stock)):
        raise ValueError(f"Неполная карточка книги: {book_url}")

    price_text = price.get_text(strip=True)
    price_match = re.fullmatch(r"£(\d+\.\d{2})", price_text)
    if price_match is None:
        raise ValueError(f"Неизвестный формат цены: {price_text}")

    rating = next(
        (RATINGS[name] for name in stars.get("class", []) if name in RATINGS),
        None,
    )
    if rating is None:
        raise ValueError(f"Неизвестный рейтинг: {book_url}")

    stock_text = stock.get_text(" ", strip=True)
    stock_match = re.search(r"(\d+)\s+available", stock_text, re.IGNORECASE)
    if stock_match:
        available = int(stock_match.group(1))
    elif "out of stock" in stock_text.lower():
        available = 0
    else:
        raise ValueError(f"Неизвестный остаток: {stock_text}")

    information = {}
    for row in table.select("tr"):
        heading, value = row.find("th"), row.find("td")
        if heading is None or value is None:
            raise ValueError(f"Некорректная строка таблицы: {book_url}")
        key = heading.get_text(strip=True)
        information[key] = value.get_text(" ", strip=True)
    if not information:
        raise ValueError(f"Пустая таблица Product Information: {book_url}")

    description = soup.select_one("#product_description + p")
    return {
        "title": title.get_text(strip=True),
        "price": float(price_match.group(1)),
        "currency": "GBP",
        "rating": rating,
        "availability": available,
        "description": (
            description.get_text(" ", strip=True) if description else ""
        ),
        "product_information": information,
        "url": book_url,
    }


In [4]:
book_url = (
    "https://books.toscrape.com/catalogue/"
    "a-light-in-the-attic_1000/index.html"
)
book = get_book_data(book_url)
assert book["title"] == "A Light in the Attic"
assert isinstance(book["price"], float)
assert 1 <= book["rating"] <= 5
book

{'title': 'A Light in the Attic',
 'price': 51.77,
 'currency': 'GBP',
 'rating': 3,
 'availability': 22,
 'description': "It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love that Silverstein. Need proof of his genius? RockabyeRockabye baby, in the treetopDon't you know a treetopIs no safe place to roc

## Задание 2. Сбор данных обо всех книгах (2 баллов)

Создайте функцию `scrape_books`, которая будет проходиться по всем страницам из каталога (вида `http://books.toscrape.com/catalogue/page-{N}.html`) и осуществлять парсинг всех страниц в цикле, используя ранее написанную `get_book_data`.

Добавьте аргумент-флаг, который будет отвечать за сохранение результата в файл: если он будет равен `True`, то информация сохранится в ту же папку в файл `books_data.txt`; иначе шаг сохранения будет пропущен.

**Также не забывайте про соблюдение PEP-8**

In [5]:
def scrape_books(
    is_save: bool = False,
    output_path: str | Path = "books_data.txt",
    start_url: str = BASE_URL,
    max_pages: int | None = None,
    delay: float = 0.2,
) -> list[dict]:
    """Пройти страницы каталога по ссылке next и собрать все карточки.

    Args:
        is_save: Сохранять ли результат; False не изменяет файлы.
        output_path: Путь к JSON-файлу, по умолчанию в текущей папке.
        start_url: Абсолютный адрес начальной страницы каталога.
        max_pages: Лимит страниц для быстрой проверки; None — весь каталог.
        delay: Пауза между запросами в секундах, неотрицательное число.

    Returns:
        Список словарей get_book_data без повторений URL книг.

    Raises:
        ValueError: Неверный аргумент, пустой каталог или цикл пагинации.
        requests.RequestException: Ошибка сети; прежняя выгрузка сохраняется.
    """
    if max_pages is not None and max_pages < 1:
        raise ValueError("max_pages: положительное число или None")
    if delay < 0:
        raise ValueError("delay не может быть отрицательным")

    books, visited_pages, visited_books = [], set(), set()
    page_url = start_url
    with create_session() as session:
        while page_url:
            if page_url in visited_pages:
                raise ValueError(f"Зацикленная пагинация: {page_url}")
            visited_pages.add(page_url)
            soup = get_soup(page_url, session)
            links = soup.select("article.product_pod h3 a[href]")
            if not links:
                raise ValueError(
                    f"В каталоге не найдены книги: {page_url}"
                )
            LOGGER.info(
                "Страница %d: найдено %d ссылок; %s",
                len(visited_pages), len(links), page_url,
            )
            for link_number, link in enumerate(links, start=1):
                book_url = urljoin(page_url, link["href"])
                if book_url not in visited_books:
                    time.sleep(delay)
                    books.append(get_book_data(book_url, session=session))
                    visited_books.add(book_url)
                if link_number % 5 == 0 or link_number == len(links):
                    LOGGER.info(
                        "Страница %d: обработано %d/%d; всего книг: %d",
                        len(visited_pages), link_number,
                        len(links), len(books),
                    )
            LOGGER.info(
                "Страниц: %d; книг: %d", len(visited_pages), len(books)
            )
            if max_pages is not None and len(visited_pages) >= max_pages:
                break
            next_link = soup.select_one("li.next a[href]")
            page_url = (
                urljoin(page_url, next_link["href"]) if next_link else None
            )
            if page_url:
                time.sleep(delay)
    if is_save:
        save_books(books, output_path)
    return books


In [6]:
# Без max_pages функция проходит весь каталог.
res = scrape_books(is_save=True, output_path=ROOT / "artifacts/books_data.txt")
print(type(res), len(res))
assert res and len({book["url"] for book in res}) == len(res)
assert all(1 <= book["rating"] <= 5 for book in res)
assert all(book["availability"] >= 0 for book in res)
saved = json.loads((ROOT / "artifacts/books_data.txt").read_text(encoding="utf-8"))
assert saved == res
print("Повторное чтение сохранённого файла: OK")
print("Первые три книги:", [book["title"] for book in res[:3]])

2026-11-15 22:21:07,713 Страница 1: найдено 20 ссылок; https://books.toscrape.com/catalogue/page-1.html


2026-11-15 22:21:09,962 Страница 1: обработано 5/20; всего книг: 5


2026-11-15 22:21:12,545 Страница 1: обработано 10/20; всего книг: 10


2026-11-15 22:21:14,860 Страница 1: обработано 15/20; всего книг: 15


2026-11-15 22:21:17,186 Страница 1: обработано 20/20; всего книг: 20


2026-11-15 22:21:17,186 Страниц: 1; книг: 20


... (291 outputs left out)

2026-11-15 22:30:32,710 Страница 50: обработано 15/20; всего книг: 995


2026-11-15 22:30:35,425 Страница 50: обработано 20/20; всего книг: 1000


2026-11-15 22:30:35,425 Страниц: 50; книг: 1000


<class 'list'> 1000
Повторное чтение сохранённого файла: OK
Первые три книги: ['A Light in the Attic', 'Tipping the Velvet', 'Soumission']


## Задание 3. Настройка регулярной выгрузки (1 баллов)

Настройте автоматический запуск функции сбора данных каждый день в 19:00.
Для автоматизации используйте библиотеку `schedule`. Функция должна запускаться в указанное время и сохранять обновленные данные в текстовый файл.



Бесконечный цикл должен обеспечивать постоянное ожидание времени для запуска задачи и выполнять ее по расписанию. Однако чтобы не перегружать систему, стоит подумать о том, чтобы выполнять проверку нужного времени не постоянно, а раз в какой-то промежуток. В этом вам может помочь `time.sleep(...)`.

Проверьте работоспособность кода локально на любом времени чч:мм.



In [7]:
def scheduled_scrape(output_path: str | Path) -> None:
    """Выгрузить каталог в output_path; записать сбой и ждать завтра."""
    try:
        books = scrape_books(is_save=True, output_path=output_path)
        LOGGER.info(
            "Выгрузка завершена: %d книг, %s", len(books), output_path
        )
    except (requests.RequestException, ValueError, OSError):
        LOGGER.exception("Выгрузка не удалась; следующий запуск по расписанию")


def create_scheduler(
    run_at: str = "19:00", output_path: str | Path = "books_data.txt"
) -> schedule.Scheduler:
    """Вернуть планировщик ежедневной выгрузки.

    Args:
        run_at: Время HH:MM (либо HH:MM:SS) в часовом поясе компьютера.
        output_path: Путь для сохранения результата ежедневного сбора.

    Returns:
        Отдельный Scheduler с одной задачей; цикл ожидания ещё не запущен.
    """
    scheduler = schedule.Scheduler()
    scheduler.every().day.at(run_at).do(scheduled_scrape, output_path)
    return scheduler


def run_scheduler(
    run_at: str = "19:00", output_path: str | Path = "books_data.txt"
) -> None:
    """Ожидать ежедневную выгрузку в run_at; остановиться по Ctrl+C."""
    scheduler = create_scheduler(run_at, output_path)
    LOGGER.info("Следующий запуск: %s", scheduler.next_run)
    try:
        while True:
            scheduler.run_pending()
            time.sleep(1)
    except KeyboardInterrupt:
        LOGGER.info("Планировщик остановлен")


output_path = ROOT / "artifacts/books_data.txt"
daily = create_scheduler("19:00", output_path)
print("Ежедневная задача:", daily.jobs[0])
print("Следующий запуск:", daily.next_run)

# Проверяем реальное срабатывание schedule через 3 секунды, сохраняя уже
# собранные данные; повторный сетевой сбор проверяется отдельно через CLI.
smoke_path = ROOT / "artifacts/schedule_smoke.txt"
smoke = schedule.Scheduler()
run_at = (datetime.now() + timedelta(seconds=3)).strftime("%H:%M:%S")
smoke_job = smoke.every().day.at(run_at).do(save_books, res, smoke_path)
deadline = time.monotonic() + 15
while smoke_job.last_run is None and time.monotonic() < deadline:
    smoke.run_pending()
    time.sleep(0.2)
assert smoke_job.last_run is not None, "Задача не сработала за 15 секунд"
assert json.loads(smoke_path.read_text(encoding="utf-8")) == res
print("Проверка сохранения по расписанию:", smoke_job.last_run, "OK")
smoke.clear()

# Бесконечный цикл находится в run_scheduler; для ручного запуска поменяйте
# флаг на True и прервите ядро, когда закончите проверку.
RUN_DAILY_SCHEDULER = False
if RUN_DAILY_SCHEDULER:
    run_scheduler("19:00", output_path)

Ежедневная задача: Job(interval=1, unit=days, do=scheduled_scrape, args=(PosixPath('/Users/andrejskakalin/Documents/botanie/proga/python/HW3/artifacts/books_data.txt'),), kwargs={})
Следующий запуск: 2026-11-16 19:00:00


Проверка сохранения по расписанию: 2026-11-15 22:30:38.160581 OK


## Задание 4. Написание автотестов (1,5 баллов)

Создайте минимум три автотеста для ключевых функций парсинга — например, `get_book_data` и `scrape_books`. Идеи проверок (можете использовать свои):

* данные о книге возвращаются в виде словаря с нужными ключами;
* список ссылок или количество собранных книг соответствует ожиданиям;
* значения отдельных полей (например, `title`) корректны.

Оформите тесты в отдельном скрипте `tests/test_scraper.py`, используйте библиотеку `pytest`. Убедитесь, что тесты проходят успешно при запуске из терминала командой `pytest`.

Также выведите результат их выполнения в ячейке ниже.

**Не забывайте про соблюдение PEP-8**


In [8]:
# Исправлен путь из исходного шаблона: tests/, а не test/.
import subprocess

completed = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests/test_scraper.py",
     "--junitxml=artifacts/pytest-results.xml"],
    cwd=ROOT, capture_output=True, text=True,
)
report = completed.stdout + completed.stderr
(ROOT / "artifacts/pytest-report.txt").write_text(report, encoding="utf-8")
print(report)
assert completed.returncode == 0, "Тесты не прошли: см. отчёт выше"

......................                                                   [100%]
- generated xml file: /Users/andrejskakalin/Documents/botanie/proga/python/HW3/artifacts/pytest-results.xml -
22 passed in 0.15s



## Задание 5. Оформление проекта на GitHub и работа с Git (3,5 баллов)

В этом задании нужно воспользоваться системой контроля версий Git и платформой GitHub для хранения и управления своим проектом. **Ссылку на свой репозиторий пришлите в форме для сдачи ответа.**

### Пошаговая инструкция и задания

**1. Установите Git на свой компьютер.**

* Для Windows: [скачайте установщик](https://git-scm.com/downloads) и выполните установку.
* Для macOS:

  ```
  brew install git
  ```
* Для Linux:

  ```
  sudo apt update
  sudo apt install git
  ```

**2. Настройте имя пользователя и email.**

Это нужно для подписи ваших коммитов, сделайте в терминале через `git config ...`.

**3. Создайте аккаунт на GitHub**, если у вас его еще нет:
[https://github.com](https://github.com)

**4. Создайте новый репозиторий на GitHub:**

* Найдите кнопку **New repository**.
* Укажите название, краткое описание, выберите тип **Public** (чтобы мы могли проверить ДЗ).
* Не ставьте галочку Initialize this repository with a README.

**5. Создайте локальную папку с проектом.** Можно в терминале, можно через UI, это не имеет значения.

**6. Инициализируйте Git в этой папке.** Здесь уже придется воспользоваться некоторой командой в терминале.

**7. Привяжите локальный репозиторий к удаленному на GitHub.**

**8. Создайте ветку разработки.** По умолчанию вы будете находиться в ветке `main`, создайте и переключитесь на ветку `hw-books-parser`.

**9. Добавьте в проект следующие файлы и папки:**

* `scraper.py` — ваш основной скрипт для сбора данных.
* `README.md` — файл с кратким описанием проекта:

  * цель;
  * инструкции по запуску;
  * список используемых библиотек.
* `requirements.txt` — файл со списком зависимостей, необходимых для проекта (не присылайте все из глобального окружения, создайте изолированную виртуальную среду, добавьте в нее все нужное для проекта и получите список библиотек через `pip freeze`).
* `artifacts/` — папка с результатами парсинга (`books_data.txt` — полностью или его часть, если весь не поместится на GitHub).
* `notebooks/` — папка с заполненным ноутбуком `HW_03_python_ds.ipynb` и запущенными ячейками с выводами на экран.
* `tests/` — папка с тестами на `pytest`, оформите их в формате скрипта(-ов) с расширением `.py`.
* `.gitignore` — стандартный файл, который позволит исключить временные файлы при добавлении в отслеживаемые (например, `__pycache__/`, `.DS_Store`, `*.pyc`, `venv/` и др.).


**10. Сделайте коммит.**

**11. Отправьте свою ветку на GitHub.**

**12. Создайте Pull Request:**

* Перейдите в репозиторий на GitHub.
* Нажмите кнопку **Compare & pull request**.
* Укажите, что было добавлено, и нажмите **Create pull request**.

**13. Выполните слияние Pull Request:**

* Убедитесь, что нет конфликтов.
* Нажмите **Merge pull request**, затем **Confirm merge**.

**14. Скачайте изменения из основной ветки локально.**



### Требования к итоговому репозиторию

* Файл `scraper.py` с рабочим кодом парсера.
* `README.md` с описанием проекта и инструкцией по запуску.
* Папка `artifacts/` с результатом сбора данных (`.txt` файл).
* Папка `tests/` с тестами на `pytest`.
* Папка `notebooks/` с заполненным ноутбуком `HW_03_python_ds_2025.ipynb`.
* Pull Request с комментарием из ветки `hw-books-parser` в ветку `main`.
* Примерная структура:

  ```
  books_scraper/
  ├── artifacts/
  │   └── books_data.txt
  ├── notebooks/
  │   └── HW_03_python_ds_2025.ipynb
  ├── scraper.py
  ├── README.md
  ├── tests/
  │   └── test_scraper.py
  ├── .gitignore
  └── requirements.txt
  ```

### Состояние задания 5

Файлы проекта и артефакты подготовлены. Репозиторий: https://github.com/ProstatusMaximus/HW3.

Историю ветки hw-books-parser, Pull Request и его слияния проверяйте в репозитории на GitHub.


## Выводы, рефлексия и сдача

### Рефлексия после выполнения (заполните здесь)

**1. Что было задано и что получилось**:

`Ответ: ...`

**2. Как выполняли**: что сработало сразу, где застряли и как выбрались:

`Ответ: ...`

**3. Использование ИИ**: цель, использованные промпты, где помощник ошибся и как вы это поняли. Если не использовали, так и напишите:

`Ответ: ...`

**4. Что осталось непонятным** (честный список, он поможет на консультации):

`Ответ: ...`

**5. Самооценка по критериям задания**:

`Ответ: ...`

**6. Дополнительный вопрос: обратная связь по заданию.** Что было ценно, что сложно, а что, на ваш взгляд, требует доработки?

`Ответ: ...`

> Напоминание: рефлексия пишется своими словами, без помощи ИИ.

